<a href="https://colab.research.google.com/github/DinethRashmikaHeshan/neuromentor-final-year-research/blob/main/learningStyleIdentifier/training/LearningStyleIdentifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files
import pandas as pd
import io

# Upload the CSV file you have
uploaded = files.upload()

# Load the data (Ensure the filename matches exactly)
file_name = list(uploaded.keys())[0]
# Changed pd.read_csv to pd.read_excel to correctly handle .xlsx files
df = pd.read_excel(io.BytesIO(uploaded[file_name]))

print("Data Loaded. Columns:", df.columns.tolist())

Saving research_data_2026-01-06.xlsx to research_data_2026-01-06 (1).xlsx
Data Loaded. Columns: ['user_id', 'task_id', 'keystrokes', 'scrolls', 'idle_time', 'compile_count', 'comment_count', 'comment_lines', 'vark_label']


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import joblib

# 1. Preprocessing
# Filter to main styles to avoid errors with single-member classes (Multimodal)
main_styles = ['Visual', 'Aural', 'Read/Write', 'Kinesthetic']
df_filtered = df[df['vark_label'].isin(main_styles)].copy()

# Feature Engineering
df_filtered['comment_ratio'] = df_filtered['comment_lines'] / (df_filtered['keystrokes'] + 1)

# Select Features
features = ['keystrokes', 'scrolls', 'idle_time', 'compile_count', 'comment_ratio']
X = df_filtered[features]
y = df_filtered['vark_label']

# Encode target labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 2. Train Model
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 3. Export for Backend
joblib.dump(model, 'vark_model.pkl')
joblib.dump(le, 'label_encoder.pkl')
# Save the list of feature names to ensure order is kept in backend
joblib.dump(features, 'feature_names.pkl')

print("Model trained and files saved locally in Colab.")

Model trained and files saved locally in Colab.


In [4]:
from google.colab import files

files.download('vark_model.pkl')
files.download('label_encoder.pkl')
files.download('feature_names.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Task
Calculate the accuracy of the trained model using the test data and display the result.

## Calculate Model Accuracy

### Subtask:
Evaluate the performance of the trained RandomForestClassifier model using the test data (X_test, y_test) and print the accuracy.


**Reasoning**:
To evaluate the model's performance, I will import the `accuracy_score` function, make predictions on the test set, and calculate the accuracy.



# Task
Improve the accuracy of the existing machine learning model by exploring other classification algorithms, optimizing hyperparameters, performing feature engineering or selection, and implementing cross-validation, then summarize the findings and report the accuracy of the best-performing model.

## Explore Different Models

### Subtask:
Experiment with other classification algorithms beyond RandomForestClassifier, such as Support Vector Machines (SVM), Gradient Boosting, or Logistic Regression, to see if they yield better results.


**Reasoning**:
I will import the necessary classifiers and accuracy score function, then train and evaluate SVC and Logistic Regression models on the test data to compare their performance.



In [9]:
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Instantiate and train SVC model
svc_model = SVC(random_state=42)
svc_model.fit(X_train, y_train)

# Instantiate and train Logistic Regression model
lr_model = LogisticRegression(random_state=42, max_iter=1000) # Increased max_iter for convergence
lr_model.fit(X_train, y_train)

# Make predictions
svc_predictions = svc_model.predict(X_test)
lr_predictions = lr_model.predict(X_test)

# Calculate and print accuracy
svc_accuracy = accuracy_score(y_test, svc_predictions)
lr_accuracy = accuracy_score(y_test, lr_predictions)

print(f"SVC Model Accuracy: {svc_accuracy:.4f}")
print(f"Logistic Regression Model Accuracy: {lr_accuracy:.4f}")

SVC Model Accuracy: 0.2500
Logistic Regression Model Accuracy: 0.2500


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


**Reasoning**:
The previous Logistic Regression model issued a `ConvergenceWarning` indicating that it did not converge even after increasing `max_iter`. Scaling the features is a common and often necessary preprocessing step for models like Logistic Regression and SVC to improve convergence and performance. I will apply `StandardScaler` to the training and test features, then re-train and evaluate both models.



In [10]:
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Instantiate and train SVC model with scaled data
svc_model = SVC(random_state=42)
svc_model.fit(X_train_scaled, y_train)

# Instantiate and train Logistic Regression model with scaled data
# Increased max_iter for convergence, and now with scaled data it should converge better
lr_model = LogisticRegression(random_state=42, max_iter=2000)
lr_model.fit(X_train_scaled, y_train)

# Make predictions
svc_predictions = svc_model.predict(X_test_scaled)
lr_predictions = lr_model.predict(X_test_scaled)

# Calculate and print accuracy
svc_accuracy = accuracy_score(y_test, svc_predictions)
lr_accuracy = accuracy_score(y_test, lr_predictions)

print(f"SVC Model Accuracy (Scaled Data): {svc_accuracy:.4f}")
print(f"Logistic Regression Model Accuracy (Scaled Data): {lr_accuracy:.4f}")

SVC Model Accuracy (Scaled Data): 0.3125
Logistic Regression Model Accuracy (Scaled Data): 0.3125


**Reasoning**:
To further explore other classification algorithms as per the subtask, I will now implement a Gradient Boosting Classifier. This will allow for comparison with the previously tested models (Random Forest, SVC, and Logistic Regression).



In [11]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

# Instantiate and train Gradient Boosting model
gbc_model = GradientBoostingClassifier(random_state=42)
gbc_model.fit(X_train_scaled, y_train)

# Make predictions
gbc_predictions = gbc_model.predict(X_test_scaled)

# Calculate and print accuracy
gbc_accuracy = accuracy_score(y_test, gbc_predictions)

print(f"Gradient Boosting Model Accuracy (Scaled Data): {gbc_accuracy:.4f}")

Gradient Boosting Model Accuracy (Scaled Data): 0.3750


## Hyperparameter Tuning

### Subtask:
Optimize the hyperparameters of the GradientBoostingClassifier model using GridSearchCV to find the best configuration for accuracy.


**Reasoning**:
To optimize the GradientBoostingClassifier, I will use GridSearchCV to systematically search for the best combination of hyperparameters, as instructed. This involves defining a parameter grid, performing cross-validation, and then evaluating the best model found.



In [12]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

# 1. Define the parameter grid for GradientBoostingClassifier
param_grid = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7]
}

# 2. Instantiate GradientBoostingClassifier
gbc = GradientBoostingClassifier(random_state=42)

# 3. Create a GridSearchCV object
grid_search = GridSearchCV(estimator=gbc, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)

# 4. Fit GridSearchCV to the scaled training data
grid_search.fit(X_train_scaled, y_train)

# 5. Print the best parameters found
print(f"Best parameters found: {grid_search.best_params_}")

# 6. Print the best cross-validation score
print(f"Best cross-validation accuracy: {grid_search.best_score_:.4f}")

# 7. Get the best estimator
best_gbc_model = grid_search.best_estimator_

# 8. Make predictions on the scaled test data using the best estimator
best_gbc_predictions = best_gbc_model.predict(X_test_scaled)

# 9. Calculate and print the accuracy of the best model on the test data
best_gbc_accuracy = accuracy_score(y_test, best_gbc_predictions)
print(f"Best Gradient Boosting Model Accuracy (Test Data): {best_gbc_accuracy:.4f}")

Best parameters found: {'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 100}
Best cross-validation accuracy: 0.4821
Best Gradient Boosting Model Accuracy (Test Data): 0.3125


## Feature Engineering/Selection

### Subtask:
Review existing features and consider creating new ones from the raw data. Also, explore feature selection methods to identify and use only the most impactful features.


**Reasoning**:
To understand which features are most influential in the `best_gbc_model` and guide potential feature selection, I will extract and print the feature importances, mapping them to their respective names and sorting them in descending order.



In [13]:
import pandas as pd

# 1. Extract feature importances from the best_gbc_model
feature_importances = best_gbc_model.feature_importances_

# 2. Create a pandas Series mapping feature names to their importance scores
feature_importance_series = pd.Series(feature_importances, index=features)

# 3. Print the feature importances in descending order
print("Feature Importances (Descending Order):")
print(feature_importance_series.sort_values(ascending=False))

# 4. Analyze these importances to identify features that might be less impactful
# (This step is observational, the code above provides the necessary data for analysis.)

Feature Importances (Descending Order):
idle_time        0.339087
comment_ratio    0.215254
scrolls          0.203808
keystrokes       0.190740
compile_count    0.051111
dtype: float64


**Reasoning**:
Based on the feature importances analysis, 'compile_count' has the lowest importance. To explore feature selection and simplify the model, I will create a new set of features by removing 'compile_count' from the existing feature list.



In [14]:
new_features = [f for f in features if f != 'compile_count']

print(f"Original features: {features}")
print(f"New feature set after removing 'compile_count': {new_features}")

Original features: ['keystrokes', 'scrolls', 'idle_time', 'compile_count', 'comment_ratio']
New feature set after removing 'compile_count': ['keystrokes', 'scrolls', 'idle_time', 'comment_ratio']


**Reasoning**:
To evaluate the impact of removing the least important feature ('compile_count'), I will prepare the dataset with the `new_features` and re-train the `GradientBoostingClassifier`. After training, I will evaluate its performance using the accuracy score to determine if feature selection improved the model.



In [15]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# 1. Select data with new_features
X_train_reduced = X_train[new_features]
X_test_reduced = X_test[new_features]

# 2. Re-scale the reduced features
scaler_reduced = StandardScaler()
X_train_scaled_reduced = scaler_reduced.fit_transform(X_train_reduced)
X_test_scaled_reduced = scaler_reduced.transform(X_test_reduced)

# 3. Instantiate and train Gradient Boosting model with reduced and scaled data
# Using the best parameters found earlier for GBC
best_gbc_model_reduced_features = GradientBoostingClassifier(
    learning_rate=grid_search.best_params_['learning_rate'],
    max_depth=grid_search.best_params_['max_depth'],
    n_estimators=grid_search.best_params_['n_estimators'],
    random_state=42
)
best_gbc_model_reduced_features.fit(X_train_scaled_reduced, y_train)

# 4. Make predictions
predictions_reduced_features = best_gbc_model_reduced_features.predict(X_test_scaled_reduced)

# 5. Calculate and print accuracy
accuracy_reduced_features = accuracy_score(y_test, predictions_reduced_features)

print(f"Gradient Boosting Model Accuracy (Reduced Features, Scaled Data): {accuracy_reduced_features:.4f}")

Gradient Boosting Model Accuracy (Reduced Features, Scaled Data): 0.3750


## Summary of Model Performance

After exploring various classification algorithms, optimizing hyperparameters, and performing feature selection, here's a summary of the model performances:

*   **RandomForestClassifier (Initial Model):** The initial RandomForestClassifier model's accuracy was calculated in a previous step, and its performance should be compared against these new results.
*   **SVC Model Accuracy (Scaled Data):** `0.3125`
*   **Logistic Regression Model Accuracy (Scaled Data):** `0.3125`
*   **Gradient Boosting Model Accuracy (Scaled Data, before tuning):** `0.3750`
*   **Gradient Boosting Model Accuracy (Scaled Data, after GridSearchCV tuning on original features):** `0.3125`
*   **Gradient Boosting Model Accuracy (Reduced Features, Scaled Data, with best hyperparameters):** `0.3750`

From these results, it appears that the **Gradient Boosting Classifier with reduced features** achieved one of the higher accuracies, matching the initial untuned Gradient Boosting Model. It's important to note that the GridSearchCV accuracy was on the validation sets, while the final test set accuracy for the best model was `0.3125`. When running the Gradient Boosting model on reduced features (and using the hyperparameters from GridSearchCV), the accuracy returned to `0.3750`.

### Best Performing Model
The best performing model in terms of accuracy on the test data is the **Gradient Boosting Classifier**, with an accuracy of **0.3750**, achieved both before hyperparameter tuning and after feature selection with the previously determined best hyperparameters.

## Cross-Validation

### Subtask:
Implement cross-validation on the best-performing model to get a more robust estimate of its performance.


**Reasoning**:
To implement cross-validation as instructed, I will import `cross_val_score`, perform 5-fold cross-validation on the `best_gbc_model_reduced_features` using the scaled reduced training data, and then calculate and print the mean and standard deviation of the accuracy scores.



In [16]:
from sklearn.model_selection import cross_val_score
import numpy as np

# Perform 5-fold cross-validation on the best-performing model with reduced features
cv_scores = cross_val_score(best_gbc_model_reduced_features, X_train_scaled_reduced, y_train, cv=5, scoring='accuracy')

# Print the accuracy scores for each fold
print(f"Cross-validation scores: {cv_scores}")

# Calculate and print the mean and standard deviation of the cross-validation scores
print(f"Mean cross-validation accuracy: {np.mean(cv_scores):.4f}")
print(f"Standard deviation of cross-validation accuracy: {np.std(cv_scores):.4f}")

Cross-validation scores: [0.53846154 0.53846154 0.46153846 0.46153846 0.33333333]
Mean cross-validation accuracy: 0.4667
Standard deviation of cross-validation accuracy: 0.0750


## Final Task

### Subtask:
Summarize the findings and the best performing model with its accuracy after attempting to improve the model.


## Summary:

### Q&A
The best performing model after attempting to improve the model is the **Gradient Boosting Classifier**, which achieved an accuracy of **0.3750** on the test data. This accuracy was observed both with the initially explored Gradient Boosting model and with the optimized Gradient Boosting model after feature selection.

### Data Analysis Key Findings
*   Initial exploration of new classification algorithms revealed that the Gradient Boosting Classifier achieved the highest accuracy of **0.3750** on scaled data, outperforming Support Vector Machines (SVC) and Logistic Regression, both of which reached an accuracy of **0.3125** after feature scaling.
*   Feature scaling using `StandardScaler` was crucial, improving the accuracy of SVC and Logistic Regression from **0.2500** to **0.3125** and resolving convergence issues for Logistic Regression.
*   Hyperparameter tuning for the Gradient Boosting Classifier using `GridSearchCV` identified optimal parameters (`learning_rate`: 0.2, `max_depth`: 3, `n_estimators`: 100), yielding a best cross-validation accuracy of **0.4821**. However, the test accuracy of the tuned model on the original features dropped to **0.3125**.
*   Feature importance analysis indicated that 'idle\_time' was the most impactful feature (importance: 0.339), while 'compile\_count' was the least impactful (importance: 0.051). Removing 'compile\_count' and retraining the Gradient Boosting Classifier with the optimized hyperparameters resulted in a test accuracy of **0.3750**, matching the untuned model's performance.
*   Cross-validation on the best-performing Gradient Boosting Classifier (with reduced features and optimized hyperparameters) provided a mean accuracy of **0.4667** across 5 folds, with a standard deviation of **0.0750**, offering a more robust estimate of performance than a single test set evaluation.

### Insights or Next Steps
*   The discrepancy between the high cross-validation accuracy (0.4821) during hyperparameter tuning and the lower test accuracy (0.3125) for the tuned model on original features suggests potential overfitting to the training data. This indicates that while the model performed well on unseen data during CV folds, its generalization to the final test set was less robust.
*   Further investigation is needed to reconcile the difference between cross-validation accuracy and test set accuracy, potentially by refining the hyperparameter search space, employing more aggressive regularization techniques, or exploring ensemble methods that leverage multiple models to reduce variance and improve generalization.


# Task
Save the `best_gbc_model_reduced_features`, the `le` (LabelEncoder), and the `new_features` list to files named 'vark_model.pkl', 'label_encoder.pkl', and 'feature_names.pkl' respectively, and then download all three files.

## Save Best Model

### Subtask:
Save the `best_gbc_model_reduced_features` (the Gradient Boosting model with optimized hyperparameters and reduced features) to a file named 'vark_model.pkl'.


**Reasoning**:
To save the best-performing model as specified in the subtask, I will use `joblib.dump()` to serialize the `best_gbc_model_reduced_features` object to a file named 'vark_model.pkl'.



In [1]:
import joblib

# Save the best Gradient Boosting model with reduced features
joblib.dump(best_gbc_model_reduced_features, 'vark_model.pkl')

print("Best model 'vark_model.pkl' saved successfully.")

NameError: name 'best_gbc_model_reduced_features' is not defined

**Reasoning**:
The `NameError` indicates that `best_gbc_model_reduced_features` was not defined because the cell where it is created (cell `60ac4112`) had not been executed. To fix this, I need to include the code from cell `60ac4112` that defines this model.



In [2]:
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier

# Re-define variables needed for best_gbc_model_reduced_features if they are not in the current kernel state.
# Assuming X_train, X_test, y_train, y_test, new_features, and grid_search are available from previous executed cells.
# If not, these would also need to be re-run or redefined.

# 1. Select data with new_features
X_train_reduced = X_train[new_features]
X_test_reduced = X_test[new_features]

# 2. Re-scale the reduced features
scaler_reduced = StandardScaler()
X_train_scaled_reduced = scaler_reduced.fit_transform(X_train_reduced)
X_test_scaled_reduced = scaler_reduced.transform(X_test_reduced)

# 3. Instantiate and train Gradient Boosting model with reduced and scaled data
# Using the best parameters found earlier for GBC
best_gbc_model_reduced_features = GradientBoostingClassifier(
    learning_rate=grid_search.best_params_['learning_rate'],
    max_depth=grid_search.best_params_['max_depth'],
    n_estimators=grid_search.best_params_['n_estimators'],
    random_state=42
)
best_gbc_model_reduced_features.fit(X_train_scaled_reduced, y_train)

# Save the best Gradient Boosting model with reduced features
joblib.dump(best_gbc_model_reduced_features, 'vark_model.pkl')

print("Best model 'vark_model.pkl' saved successfully.")

NameError: name 'X_train' is not defined

**Reasoning**:
The previous code failed because several critical variables like `X_train`, `y_train`, `new_features`, `le`, and `grid_search` were not defined in the current execution environment. To fix this, I need to re-run all the preceding necessary steps from the notebook, including data loading, preprocessing, train-test splitting, feature engineering, scaling, and hyperparameter tuning, to ensure all required variables are properly initialized before attempting to save the model and related objects.



In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
import joblib
import io

# --- 1. Re-load the data and initial preprocessing (from SR1Gh-7lneGY and AQG7-Bpzn8Yt) ---
# Assuming `uploaded` from cell SR1Gh-7lneGY still holds the file content or re-upload if needed.
# For reproducibility, let's assume `df` is already in memory or needs to be loaded again.
# If `uploaded` is not available, this part would fail. Let's make it robust.

# Since the traceback shows X_train not defined, implying a fresh kernel or out-of-order execution,
# we need to re-execute all dependencies that define the variables.

# This block would typically be run after data upload, but to make this cell self-contained:
# For demonstration, assuming a dummy df if actual upload is not possible in this isolated block.
# In a real Colab session, you'd ensure the previous cells executed.

# Re-create df from the last known state of df in the notebook (after upload in SR1Gh-7lneGY)
# If df is not in scope, this line would need the actual file upload mechanism or a pre-loaded df.
# For now, we'll assume df is defined from previous executed cells or from the original data loading.

# Filter to main styles (from AQG7-Bpzn8Yt)
main_styles = ['Visual', 'Aural', 'Read/Write', 'Kinesthetic']
df_filtered = df[df['vark_label'].isin(main_styles)].copy()

# Feature Engineering (from AQG7-Bpzn8Yt)
df_filtered['comment_ratio'] = df_filtered['comment_lines'] / (df_filtered['keystrokes'] + 1)

# Select Features (from AQG7-Bpzn8Yt)
features = ['keystrokes', 'scrolls', 'idle_time', 'compile_count', 'comment_ratio']
X = df_filtered[features]
y = df_filtered['vark_label']

# Encode target labels (from AQG7-Bpzn8Yt)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split data (from AQG7-Bpzn8Yt)
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# --- 2. Scaling for GridSearchCV (from a2aae158) ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- 3. Run GridSearchCV to get `grid_search` (from fd64203e) ---
param_grid = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7]
}
gbc = GradientBoostingClassifier(random_state=42)
grid_search = GridSearchCV(estimator=gbc, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

# --- 4. Define `new_features` (from 57cbd6ea) ---
new_features = [f for f in features if f != 'compile_count']

# --- 5. Re-create and train `best_gbc_model_reduced_features` (from 60ac4112) ---
# Select data with new_features
X_train_reduced = X_train[new_features]
X_test_reduced = X_test[new_features]

# Re-scale the reduced features
scaler_reduced = StandardScaler() # A new scaler is used for reduced features as per original notebook
X_train_scaled_reduced = scaler_reduced.fit_transform(X_train_reduced)
X_test_scaled_reduced = scaler_reduced.transform(X_test_reduced)

# Instantiate and train Gradient Boosting model with reduced and scaled data
best_gbc_model_reduced_features = GradientBoostingClassifier(
    learning_rate=grid_search.best_params_['learning_rate'],
    max_depth=grid_search.best_params_['max_depth'],
    n_estimators=grid_search.best_params_['n_estimators'],
    random_state=42
)
best_gbc_model_reduced_features.fit(X_train_scaled_reduced, y_train)

# --- 6. Save all necessary files ---
# Save the best Gradient Boosting model with reduced features
joblib.dump(best_gbc_model_reduced_features, 'vark_model.pkl')
print("Best model 'vark_model.pkl' saved successfully.")

# Save the LabelEncoder (le)
joblib.dump(le, 'label_encoder.pkl')
print("Label encoder 'label_encoder.pkl' saved successfully.")

# Save the new_features list
joblib.dump(new_features, 'feature_names.pkl')
print("Feature names 'feature_names.pkl' saved successfully.")


NameError: name 'df' is not defined

**Reasoning**:
The previous code failed because the `df` DataFrame was not defined, indicating that the initial data loading step was missed or the variable was not retained in the environment. To resolve this, I need to re-include the data loading and initial preprocessing steps from the beginning of the notebook to ensure `df` and all subsequent dependent variables are correctly initialized before proceeding with model training and saving.



In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
import joblib
import io
from google.colab import files

# --- 1. Re-load the data and initial preprocessing (from SR1Gh-7lneGY and AQG7-Bpzn8Yt) ---
# The original data loading involved files.upload(). Assuming the file 'research_data_2026-01-06.xlsx'
# is still present in the Colab environment from previous uploads.

# Hardcoding the file name based on previous execution output for reproducibility in this block.
file_name = 'research_data_2026-01-06.xlsx' # This should match the name of the uploaded file
df = pd.read_excel(file_name)

# Filter to main styles (from AQG7-Bpzn8Yt)
main_styles = ['Visual', 'Aural', 'Read/Write', 'Kinesthetic']
df_filtered = df[df['vark_label'].isin(main_styles)].copy()

# Feature Engineering (from AQG7-Bpzn8Yt)
df_filtered['comment_ratio'] = df_filtered['comment_lines'] / (df_filtered['keystrokes'] + 1)

# Select Features (from AQG7-Bpzn8Yt)
features = ['keystrokes', 'scrolls', 'idle_time', 'compile_count', 'comment_ratio']
X = df_filtered[features]
y = df_filtered['vark_label']

# Encode target labels (from AQG7-Bpzn8Yt)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split data (from AQG7-Bpzn8Yt)
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# --- 2. Scaling for GridSearchCV (from a2aae158) ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- 3. Run GridSearchCV to get `grid_search` (from fd64203e) ---
param_grid = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7]
}
gbc = GradientBoostingClassifier(random_state=42)
grid_search = GridSearchCV(estimator=gbc, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

# --- 4. Define `new_features` (from 57cbd6ea) ---
new_features = [f for f in features if f != 'compile_count']

# --- 5. Re-create and train `best_gbc_model_reduced_features` (from 60ac4112) ---
# Select data with new_features
X_train_reduced = X_train[new_features]
X_test_reduced = X_test[new_features]

# Re-scale the reduced features
scaler_reduced = StandardScaler() # A new scaler is used for reduced features as per original notebook
X_train_scaled_reduced = scaler_reduced.fit_transform(X_train_reduced)
X_test_scaled_reduced = scaler_reduced.transform(X_test_reduced)

# Instantiate and train Gradient Boosting model with reduced and scaled data
best_gbc_model_reduced_features = GradientBoostingClassifier(
    learning_rate=grid_search.best_params_['learning_rate'],
    max_depth=grid_search.best_params_['max_depth'],
    n_estimators=grid_search.best_params_['n_estimators'],
    random_state=42
)
best_gbc_model_reduced_features.fit(X_train_scaled_reduced, y_train)

# --- 6. Save all necessary files ---
# Save the best Gradient Boosting model with reduced features
joblib.dump(best_gbc_model_reduced_features, 'vark_model.pkl')
print("Best model 'vark_model.pkl' saved successfully.")

# Save the LabelEncoder (le)
joblib.dump(le, 'label_encoder.pkl')
print("Label encoder 'label_encoder.pkl' saved successfully.")

# Save the new_features list
joblib.dump(new_features, 'feature_names.pkl')
print("Feature names 'feature_names.pkl' saved successfully.")

FileNotFoundError: [Errno 2] No such file or directory: 'research_data_2026-01-06.xlsx'

**Reasoning**:
The `FileNotFoundError` indicates that `pd.read_excel(file_name)` failed because the Excel file was not found in the current environment. This typically happens when the `files.upload()` operation (which is interactive) needs to be re-executed, or when the file is not saved to disk as expected. To ensure the DataFrame `df` is correctly loaded, I need to re-incorporate the original `files.upload()` mechanism and read the uploaded file content using `io.BytesIO`, which handles the file directly from the `uploaded` dictionary.



In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
import joblib
import io
from google.colab import files

# --- 1. Re-load the data and initial preprocessing (from SR1Gh-7lneGY and AQG7-Bpzn8Yt) ---

# Re-execute the file upload step to ensure the data is available.
# This will require interactive user input if the file is not already uploaded in the current session.
print("Please upload your Excel file (e.g., 'research_data_2026-01-06.xlsx') again if prompted.")
uploaded = files.upload()

# Load the data from the uploaded content using io.BytesIO
file_name = list(uploaded.keys())[0]
df = pd.read_excel(io.BytesIO(uploaded[file_name]))

# Filter to main styles (from AQG7-Bpzn8Yt)
main_styles = ['Visual', 'Aural', 'Read/Write', 'Kinesthetic']
df_filtered = df[df['vark_label'].isin(main_styles)].copy()

# Feature Engineering (from AQG7-Bpzn8Yt)
df_filtered['comment_ratio'] = df_filtered['comment_lines'] / (df_filtered['keystrokes'] + 1)

# Select Features (from AQG7-Bpzn8Yt)
features = ['keystrokes', 'scrolls', 'idle_time', 'compile_count', 'comment_ratio']
X = df_filtered[features]
y = df_filtered['vark_label']

# Encode target labels (from AQG7-Bpzn8Yt)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split data (from AQG7-Bpzn8Yt)
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# --- 2. Scaling for GridSearchCV (from a2aae158) ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- 3. Run GridSearchCV to get `grid_search` (from fd64203e) ---
param_grid = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7]
}
gbc = GradientBoostingClassifier(random_state=42)
grid_search = GridSearchCV(estimator=gbc, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

# --- 4. Define `new_features` (from 57cbd6ea) ---
new_features = [f for f in features if f != 'compile_count']

# --- 5. Re-create and train `best_gbc_model_reduced_features` (from 60ac4112) ---
# Select data with new_features
X_train_reduced = X_train[new_features]
X_test_reduced = X_test[new_features]

# Re-scale the reduced features
scaler_reduced = StandardScaler() # A new scaler is used for reduced features as per original notebook
X_train_scaled_reduced = scaler_reduced.fit_transform(X_train_reduced)
X_test_scaled_reduced = scaler_reduced.transform(X_test_reduced)

# Instantiate and train Gradient Boosting model with reduced and scaled data
best_gbc_model_reduced_features = GradientBoostingClassifier(
    learning_rate=grid_search.best_params_['learning_rate'],
    max_depth=grid_search.best_params_['max_depth'],
    n_estimators=grid_search.best_params_['n_estimators'],
    random_state=42
)
best_gbc_model_reduced_features.fit(X_train_scaled_reduced, y_train)

# --- 6. Save all necessary files ---
# Save the best Gradient Boosting model with reduced features
joblib.dump(best_gbc_model_reduced_features, 'vark_model.pkl')
print("Best model 'vark_model.pkl' saved successfully.")

# Save the LabelEncoder (le)
joblib.dump(le, 'label_encoder.pkl')
print("Label encoder 'label_encoder.pkl' saved successfully.")

# Save the new_features list
joblib.dump(new_features, 'feature_names.pkl')
print("Feature names 'feature_names.pkl' saved successfully.")

Please upload your Excel file (e.g., 'research_data_2026-01-06.xlsx') again if prompted.


Saving research_data_2026-01-06.xlsx to research_data_2026-01-06.xlsx
Best model 'vark_model.pkl' saved successfully.
Label encoder 'label_encoder.pkl' saved successfully.
Feature names 'feature_names.pkl' saved successfully.


**Reasoning**:
The subtask requires downloading the saved files. I will use `files.download()` for each of the three files: 'vark_model.pkl', 'label_encoder.pkl', and 'feature_names.pkl'.



In [6]:
from google.colab import files

files.download('vark_model.pkl')
files.download('label_encoder.pkl')
files.download('feature_names.pkl')

print("Files downloaded successfully.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Files downloaded successfully.


## Final Task

### Subtask:
Confirm that the best model, label encoder, and feature names have been successfully saved and downloaded.


## Summary:

### Q&A
The best model (`best_gbc_model_reduced_features`), label encoder (`le`), and feature names (`new_features`) were successfully saved to 'vark_model.pkl', 'label_encoder.pkl', and 'feature_names.pkl' respectively, and all three files were successfully downloaded.

### Data Analysis Key Findings
*   Initial attempts to save the model and related objects failed due to `NameError`, indicating that variables from previous execution steps were not persistent in the current environment.
*   Loading the data directly with `pd.read_excel('filename.xlsx')` resulted in a `FileNotFoundError`; in Google Colab, uploaded files require handling via `io.BytesIO` to access their content.
*   A comprehensive re-execution of all preceding data loading, preprocessing, feature engineering, and model training steps was required to correctly define all necessary variables and train the final model.
*   After re-executing all preceding steps, the `best_gbc_model_reduced_features`, the `le` (LabelEncoder), and the `new_features` list were successfully saved to 'vark_model.pkl', 'label_encoder.pkl', and 'feature_names.pkl', respectively.
*   All three saved files were successfully downloaded.

### Insights or Next Steps
*   When working in interactive environments like Google Colab, ensure all necessary data loading, preprocessing, and model training steps are re-executed within the same session or script to maintain variable definitions and avoid `NameError`s.
*   For future tasks involving user-uploaded files in Google Colab, remember to use `io.BytesIO` in conjunction with `files.upload()` for correct file content access by libraries like pandas.
